# Stanford RNA 3D Folding 2 — Submission NotebookThis notebook predicts RNA 3D structures for the[Stanford RNA 3D Folding 2](https://www.kaggle.com/competitions/stanford-rna-3d-folding-2)Kaggle competition using the **friendly-telegram** RNA structure prediction toolkit.**Strategy**:1. Template matching — find the closest training RNA by sequence k-mer similarity2. Ab-initio torsion-space diffusion — for targets without a good template3. BPP-guided coordinate refinement — improves ab-initio predictions4. Topology correction — fixes backbone chain breaks5. Ensemble diversification — produces 5 conformers per target

In [ ]:
import sys, os, time, warnings, gc
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Detect environment and paths ──────────────────────────────────────────
KAGGLE = os.path.exists('/kaggle')

# Repository code (added as a Kaggle dataset)
REPO_PATH = None
for p in ['/kaggle/input/friendly-telegram',
          '/kaggle/input/friendly-telegram/friendly-telegram',
          os.path.dirname(os.path.abspath('__file__')),
          '.']:
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, 'rna_dataset.py')):
        REPO_PATH = p
        break
if REPO_PATH is None:
    raise FileNotFoundError("Repository code not found — add 'friendly-telegram' as a Kaggle dataset")
sys.path.insert(0, REPO_PATH)

# Competition data
DATA_PATH = None
for p in ['/kaggle/input/stanford-rna-3d-folding-2',
          '/kaggle/input/stanford-rna-3d-folding',
          os.path.join('.', 'data', 'stanford-rna-3d-folding-2'),
          os.path.join('.', 'data', 'stanford-rna-3d-folding')]:
    if os.path.isdir(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Competition data not found — add 'stanford-rna-3d-folding-2' as data source")

OUTPUT_DIR = '/kaggle/working' if KAGGLE else '.'

print(f"Repository : {REPO_PATH}")
print(f"Data       : {DATA_PATH}")
print(f"Output     : {OUTPUT_DIR}")
print(f"Data files : {sorted(os.listdir(DATA_PATH))}")

In [ ]:
# ── Load test sequences ────────────────────────────────────────────────────
test_seq_path = os.path.join(DATA_PATH, 'test_sequences.csv')
test_df = pd.read_csv(test_seq_path)
print(f"Test dataframe: {test_df.shape}")
print(test_df.head(3))

def _detect_col(df, candidates, fallback_idx=0):
    for c in candidates:
        if c in df.columns:
            return c
    return df.columns[fallback_idx]

id_col  = _detect_col(test_df, ['target_id', 'ID', 'id'])
seq_col = _detect_col(test_df, ['sequence', 'seq'], fallback_idx=1)

test_targets = {}
for _, row in test_df.iterrows():
    test_targets[str(row[id_col])] = str(row[seq_col])

print(f"\nTest targets : {len(test_targets)}")
lengths = [len(s) for s in test_targets.values()]
print(f"Length range  : {min(lengths)} – {max(lengths)} nt")

# ── Load sample submission for output format ──────────────────────────────
sample_sub_path = os.path.join(DATA_PATH, 'sample_submission.csv')
sample_sub = pd.read_csv(sample_sub_path) if os.path.isfile(sample_sub_path) else None
if sample_sub is not None:
    print(f"\nSample submission: {sample_sub.shape}")
    print(f"Columns: {list(sample_sub.columns)}")
    print(sample_sub.head(2))
else:
    print("\nNo sample_submission.csv found — will use default format")

In [ ]:
from rna_dataset import _parse_sequences, _parse_labels

train_db = {}

# ── Sequences ──
train_seq_path = os.path.join(DATA_PATH, 'train_sequences.csv')
if os.path.isfile(train_seq_path):
    tsdf = pd.read_csv(train_seq_path)
    for tid, seq in _parse_sequences(tsdf).items():
        train_db[tid] = {'sequence': seq, 'coords': None}
    del tsdf
    print(f"Training sequences : {len(train_db)}")
else:
    print("No training sequences found")

# ── Labels (coordinates) ──
train_lab_path = os.path.join(DATA_PATH, 'train_labels.csv')
if os.path.isfile(train_lab_path):
    print("Loading training labels …")
    t0 = time.time()
    tldf = pd.read_csv(train_lab_path)
    train_coords = _parse_labels(tldf)
    for tid, coords in train_coords.items():
        if tid in train_db:
            train_db[tid]['coords'] = coords
        else:
            train_db[tid] = {'sequence': '', 'coords': coords}
    del tldf, train_coords
    gc.collect()
    print(f"  Loaded in {time.time()-t0:.1f}s")
else:
    print("No training labels found")

n_with_coords = sum(1 for v in train_db.values() if v['coords'] is not None)
print(f"\nTemplate database : {len(train_db)} sequences, {n_with_coords} with 3-D coords")

In [ ]:
# ── Warm up Numba JIT to avoid compilation overhead during prediction ─────
print("Warming up JIT …", end=" ", flush=True)
t0 = time.time()

try:
    from whitebox.partition_function import compute_bpp_linear_approx
    from whitebox.riemannian_backbone import torsion_diffusion_step
    from modules.topology_correction import TopologyCorrector
    from pipeline.full_pipeline import _sequence_to_int8, _torsions_to_coords

    _dummy_seq = np.array([0,1,2,3,0,1,2,3,0,1], dtype=np.int8)
    _ = compute_bpp_linear_approx(_dummy_seq, len(_dummy_seq))
    _dummy_t = np.zeros((10, 7), dtype=np.float32)
    _ = torsion_diffusion_step(_dummy_t, np.float32(0.5), np.float32(10.0))
    _ = _torsions_to_coords(_dummy_t)
    _tc = TopologyCorrector()
    _ = _tc(np.random.randn(10, 3).astype(np.float32))
    print(f"done ({time.time()-t0:.1f}s)")
    HAVE_PIPELINE = True
except Exception as e:
    print(f"\n  Pipeline warm-up failed: {e}")
    HAVE_PIPELINE = False

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Helper functions
# ═══════════════════════════════════════════════════════════════════════════

# Minimum k-mer Jaccard score to accept a template match
TEMPLATE_SCORE_THRESHOLD = 0.08

def kmer_set(seq, k=3):
    """Return the set of k-mers for a sequence."""
    return set(seq[i:i+k] for i in range(max(0, len(seq) - k + 1)))


def kmer_similarity(seq1, seq2, k=3):
    """Jaccard similarity of k-mer sets."""
    k1, k2 = kmer_set(seq1, k), kmer_set(seq2, k)
    if not k1 or not k2:
        return 0.0
    return len(k1 & k2) / len(k1 | k2)


def find_best_template(test_seq, train_db, k=3):
    """Return (target_id, score) for the best training template."""
    best_tid, best_score = None, 0.0
    tlen = len(test_seq)
    for tid, info in train_db.items():
        if info['coords'] is None or not info['sequence']:
            continue
        slen = len(info['sequence'])
        ratio = min(tlen, slen) / max(tlen, slen) if max(tlen, slen) > 0 else 0
        if ratio < 0.25:
            continue
        sim = kmer_similarity(test_seq, info['sequence'], k)
        score = sim * (0.5 + 0.5 * ratio)
        if score > best_score:
            best_score, best_tid = score, tid
    return best_tid, best_score


def resize_coords_arclength(coords, target_len):
    """Resize a (L_src, 3) coordinate array to (target_len, 3) via arc-length
    parameterised interpolation.  Preserves the overall backbone shape."""
    src_len = coords.shape[0]
    if src_len == target_len:
        return coords.copy().astype(np.float32)
    if src_len < 2:
        return np.zeros((target_len, 3), dtype=np.float32)

    # Cumulative arc length along the backbone
    diffs = np.diff(coords.astype(np.float64), axis=0)
    seg_len = np.sqrt(np.sum(diffs ** 2, axis=1))
    cumulative_arclength = np.concatenate([[0.0], np.cumsum(seg_len)])
    total = cumulative_arclength[-1]
    if total < 1e-8:
        return np.tile(coords[0].astype(np.float32), (target_len, 1))
    cumulative_arclength /= total          # normalise to [0, 1]
    t = np.linspace(0.0, 1.0, target_len)  # target positions
    out = np.zeros((target_len, 3), dtype=np.float32)
    for d in range(3):
        out[:, d] = np.interp(t, cumulative_arclength,
                               coords[:, d].astype(np.float64)).astype(np.float32)
    return out


def fill_nan_coords(coords_2d):
    """Replace NaN positions by linear interpolation along the backbone."""
    c = coords_2d.copy()
    for d in range(3):
        col = c[:, d]
        valid = ~np.isnan(col)
        if valid.sum() < 2:
            col[:] = 0.0
        elif not valid.all():
            col[~valid] = np.interp(np.where(~valid)[0],
                                     np.where(valid)[0],
                                     col[valid])
        c[:, d] = col
    return c


print("Helpers defined ✓")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Prediction functions
# ═══════════════════════════════════════════════════════════════════════════

def predict_from_template(test_seq, template_coords, n_conformers=5):
    """Return *n_conformers* coordinate arrays (L_test, 3) derived from a
    training template's (L_train, 5, 3) coordinate tensor."""
    L = len(test_seq)
    n_avail = template_coords.shape[1]
    conformers = []

    for c in range(n_conformers):
        # Pick source conformer (cycle through available, with fallback to 0)
        src = c % n_avail
        tc = template_coords[:, src, :].copy()

        # Handle NaN: try other conformers then interpolate
        if np.any(np.isnan(tc)):
            for alt in range(n_avail):
                candidate = template_coords[:, alt, :]
                if not np.any(np.isnan(candidate)):
                    tc = candidate.copy()
                    break
            else:
                tc = fill_nan_coords(tc)

        # Resize to test-sequence length via arc-length interpolation
        fitted = resize_coords_arclength(tc, L)

        # Add random perturbation for diversity (conformer 0 = unperturbed).
        # sqrt scaling keeps noise bounded for higher conformer indices.
        if c > 0:
            noise_scale = 0.3 * np.sqrt(c)
            fitted = fitted + np.random.randn(L, 3).astype(np.float32) * noise_scale

        conformers.append(fitted.astype(np.float32))

    return conformers


def bpp_guided_refinement(coords, seq_int8, n_steps=50, lr=0.005):
    """Refine (L, 3) coordinates using base-pair probability restraints and
    backbone connectivity constraints.

    Target distances are derived from RNA crystallographic statistics:
      - Base-paired residues: ~8 Å (C3'-C3' distance across a canonical pair)
      - Consecutive backbone C3' atoms: ~3.8 Å (standard A-form helix spacing)
    """
    from whitebox.partition_function import compute_bpp_linear_approx

    L = coords.shape[0]
    bpp = compute_bpp_linear_approx(seq_int8, L)

    # Pre-compute significant pairs (BPP > 0.25)
    pairs = []
    for i in range(L):
        for j in range(i + 4, L):
            if bpp[i, j] > 0.25:
                pairs.append((i, j, float(bpp[i, j])))

    c = coords.astype(np.float64).copy()
    target_pair_dist = 8.0   # Å — typical C3'-C3' base-pair distance
    target_bond_dist = 3.8   # Å — standard A-form backbone spacing

    for _ in range(n_steps):
        forces = np.zeros_like(c)

        # Pair restraints
        for i, j, w in pairs:
            diff = c[j] - c[i]
            d = np.sqrt(np.sum(diff ** 2)) + 1e-8
            f = w * (d - target_pair_dist) * (diff / d)
            forces[i] += f * lr
            forces[j] -= f * lr

        # Backbone connectivity restraints
        for i in range(L - 1):
            diff = c[i + 1] - c[i]
            d = np.sqrt(np.sum(diff ** 2)) + 1e-8
            f = 5.0 * (d - target_bond_dist) * (diff / d)
            forces[i]     += f * lr
            forces[i + 1] -= f * lr

        c += forces

    return c.astype(np.float32)


def ab_initio_predict(sequence, n_conformers=5, n_candidates=40):
    """Generate conformers ab initio via torsion-space diffusion, optionally
    refined with BPP restraints."""
    from pipeline.full_pipeline import _sequence_to_int8, _torsions_to_coords
    from whitebox.riemannian_backbone import torsion_diffusion_step
    from modules.topology_correction import TopologyCorrector

    L = len(sequence)
    seq_int8 = _sequence_to_int8(sequence)
    corrector = TopologyCorrector(threshold=4.5)

    candidates = []
    for i in range(n_candidates):
        torsions = np.zeros((L, 7), dtype=np.float32)
        noise_scale = np.float32(0.05 + 0.95 * i / max(n_candidates - 1, 1))
        torsions = torsion_diffusion_step(torsions, noise_scale, np.float32(10.0))
        coords = _torsions_to_coords(torsions)
        coords = corrector(coords)
        # BPP-guided refinement (skip for very long sequences to save time)
        if L <= 500:
            try:
                coords = bpp_guided_refinement(coords, seq_int8, n_steps=30, lr=0.003)
                coords = corrector(coords)
            except Exception:
                pass
        candidates.append(coords)

    if len(candidates) <= n_conformers:
        return candidates[:n_conformers]

    # Greedy diversity selection (max-min distance)
    selected = [0]
    for _ in range(n_conformers - 1):
        best_j, best_d = 0, -1.0
        for j in range(len(candidates)):
            if j in selected:
                continue
            md = min(
                np.mean(np.linalg.norm(candidates[j] - candidates[s], axis=1))
                for s in selected
            )
            if md > best_d:
                best_d, best_j = md, j
        selected.append(best_j)

    return [candidates[i] for i in selected]


print("Prediction functions defined ✓")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Main prediction loop
# ═══════════════════════════════════════════════════════════════════════════
from modules.topology_correction import TopologyCorrector

corrector = TopologyCorrector(threshold=4.5)
predictions = {}
method_counts = {'template': 0, 'ab_initio': 0}

print(f"Predicting {len(test_targets)} targets …")
overall_t0 = time.time()

for idx, (tid, seq) in enumerate(test_targets.items()):
    t0 = time.time()
    L = len(seq)

    # ── Try template-based prediction ─────────────────────────────────────
    best_tid, best_score = find_best_template(seq, train_db)

    if best_tid is not None and best_score > TEMPLATE_SCORE_THRESHOLD and train_db[best_tid]['coords'] is not None:
        conformers = predict_from_template(seq, train_db[best_tid]['coords'])
        method_counts['template'] += 1
        method = 'T'
    elif HAVE_PIPELINE:
        conformers = ab_initio_predict(seq)
        method_counts['ab_initio'] += 1
        method = 'A'
    else:
        # Minimal fallback: helical trace
        conformers = []
        for c in range(5):
            coords = np.zeros((L, 3), dtype=np.float32)
            for i in range(L):
                angle = i * 0.6 + c * 0.3
                coords[i] = [3.8 * i * np.cos(angle * 0.15),
                              3.8 * i * np.sin(angle * 0.15),
                              3.4 * np.sin(angle)]
            conformers.append(coords)
        method_counts['ab_initio'] += 1
        method = 'H'

    # ── Ensure exactly 5 conformers ───────────────────────────────────────
    while len(conformers) < 5:
        base = conformers[-1].copy()
        conformers.append(base + np.random.randn(L, 3).astype(np.float32) * 0.5)
    conformers = conformers[:5]

    # ── Topology correction ───────────────────────────────────────────────
    fixed = []
    for c in conformers:
        c = np.asarray(c, dtype=np.float32)
        if c.shape != (L, 3):
            c = resize_coords_arclength(c, L) if c.shape[0] != L else c[:, :3]
        c = corrector(c)
        fixed.append(c)

    predictions[tid] = fixed

    if (idx + 1) % max(1, len(test_targets) // 10) == 0 or idx == 0:
        print(f"  [{idx+1:>4}/{len(test_targets)}] {tid:>12} L={L:>4}  "
              f"method={method}  score={best_score:.3f}  {time.time()-t0:.1f}s")

elapsed = time.time() - overall_t0
print(f"\nDone — {len(predictions)} targets in {elapsed:.1f}s")
print(f"Methods: {method_counts}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Build and save submission.csv
# ═══════════════════════════════════════════════════════════════════════════

# ── Detect expected column format ─────────────────────────────────────────
if sample_sub is not None:
    sub_columns = list(sample_sub.columns)
    id_col_name = sub_columns[0]
    coord_cols_sample = [c for c in sub_columns if c != id_col_name]
else:
    id_col_name = 'id'
    coord_cols_sample = []
    for ci in range(1, 6):
        coord_cols_sample.extend([f'x_{ci}', f'y_{ci}', f'z_{ci}'])
    sub_columns = [id_col_name] + coord_cols_sample

print(f"Submission columns ({len(sub_columns)}): {sub_columns[:6]} …")

# ── Build rows ────────────────────────────────────────────────────────────
rows = []
for tid in test_targets:
    seq = test_targets[tid]
    conformers = predictions[tid]
    L = len(seq)

    for resid in range(L):
        row = {id_col_name: f"{tid}_{resid + 1}"}
        row['resname'] = seq[resid]
        row['resid'] = resid + 1
        for ci in range(5):
            cn = ci + 1
            row[f'x_{cn}'] = round(float(conformers[ci][resid, 0]), 4)
            row[f'y_{cn}'] = round(float(conformers[ci][resid, 1]), 4)
            row[f'z_{cn}'] = round(float(conformers[ci][resid, 2]), 4)
        rows.append(row)

submission = pd.DataFrame(rows)

# ── Align to sample submission columns (handles differing col order) ──────
if sample_sub is not None:
    for col in sub_columns:
        if col not in submission.columns:
            submission[col] = 0.0
    submission = submission[sub_columns]

out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"Shape: {submission.shape}")
print(submission.head(3))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cross-validation on training data (estimate expected TM-score)
# ═══════════════════════════════════════════════════════════════════════════
try:
    from metrics.tm_score import tm_score, kabsch_align

    rng = np.random.default_rng(123)
    val_tids = [tid for tid in train_db
                if train_db[tid]['coords'] is not None
                and train_db[tid]['sequence']
                and not np.any(np.isnan(train_db[tid]['coords'][:, 0, :]))]
    if len(val_tids) > 30:
        val_tids = list(rng.choice(val_tids, 30, replace=False))

    val_scores = []
    for tid in val_tids:
        true = train_db[tid]['coords'][:, 0, :]  # first conformer
        seq  = train_db[tid]['sequence']
        L    = len(seq)
        if true.shape[0] != L:
            continue

        # Leave-one-out template search
        temp_db = {k: v for k, v in train_db.items() if k != tid}
        bt, bs = find_best_template(seq, temp_db)

        if bt is not None and bs > TEMPLATE_SCORE_THRESHOLD and temp_db[bt]['coords'] is not None:
            pred = predict_from_template(seq, temp_db[bt]['coords'], n_conformers=1)[0]
        else:
            continue  # skip ab-initio for speed

        aligned = kabsch_align(pred, true)
        sc = tm_score(aligned, true)
        val_scores.append(sc)

    if val_scores:
        arr = np.array(val_scores)
        print(f"Validation TM-scores (n={len(arr)}):")
        print(f"  mean  = {arr.mean():.4f}")
        print(f"  median= {np.median(arr):.4f}")
        print(f"  >0.5  = {(arr > 0.5).sum()}/{len(arr)}")
        print(f"  >0.7  = {(arr > 0.7).sum()}/{len(arr)}")
    else:
        print("No validation targets available for scoring.")
except Exception as e:
    print(f"Validation skipped: {e}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Final submission sanity checks
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("Submission sanity checks")
print("=" * 60)

sub = pd.read_csv(os.path.join(OUTPUT_DIR, 'submission.csv'))
sub_ids = set(sub.iloc[:, 0].apply(lambda x: str(x).rsplit('_', 1)[0]))
test_ids = set(test_targets.keys())
missing = test_ids - sub_ids
print(f"Targets in test set    : {len(test_ids)}")
print(f"Targets in submission  : {len(sub_ids)}")
if missing:
    print(f"  ⚠ Missing targets: {missing}")
else:
    print("  ✓ All targets present")

expected_rows = sum(len(s) for s in test_targets.values())
print(f"Expected rows          : {expected_rows}")
print(f"Actual rows            : {len(sub)}")
assert len(sub) == expected_rows, f"Row count mismatch: {len(sub)} vs {expected_rows}"
print("  ✓ Row count correct")

coord_cols = [c for c in sub.columns if c.startswith(('x_', 'y_', 'z_'))]
vals = sub[coord_cols].values
print(f"NaN count              : {np.isnan(vals).sum()}")
print(f"Inf count              : {np.isinf(vals).sum()}")
assert np.isnan(vals).sum() == 0, "NaN values in submission!"
assert np.isinf(vals).sum() == 0, "Inf values in submission!"
print("  ✓ No NaN / Inf values")

print("\n✅  submission.csv is ready for upload!")